<a href="https://colab.research.google.com/github/ajamil05/AI-Automated-Recuritment-System/blob/main/AI_Automated_Recuitment_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Job Description Parsing

In [58]:
job_title = input('what job would you like to apply to')
typeofjob = input('what type of job would you like to apply to:\n1. Remote\n2. Onsite\n3. Hybrid\n')
Location = input('what location would you like to apply to')


what job would you like to apply toComputer Science
what type of job would you like to apply to:
1. Remote
2. Onsite
3. Hybrid
1
what location would you like to apply toLondon


In [59]:
import requests
from bs4 import BeautifulSoup
import json
import re
import random

N8N_JOB_WEBHOOK_URL = "https://bagged-attest-snooper.ngrok-free.dev/webhook/job-description"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

# Dynamic input parsing from your input cell
clean_job_title = job_title.strip() if 'job_title' in globals() else "AI Intern"
clean_location = Location.strip() if 'Location' in globals() else "Manchester"
clean_typeofjob = typeofjob.strip() if 'typeofjob' in globals() else "1"

# Map the numeric option in typeofjob to a human-readable string
job_type_mapping = {"1": "Remote", "2": "Onsite", "3": "Hybrid"}
job_type_str = job_type_mapping.get(clean_typeofjob, "Remote")

# Dynamic Fallback Search Strategies based on YOUR input
words = clean_job_title.split()
broader_title = words[-1] if len(words) > 1 else clean_job_title

search_strategies = [
    # 1. Preferred exact search
    {"keywords": clean_job_title, "location": clean_location, "f_WT": clean_typeofjob},
    # 2. Broadened search keyword from user input
    {"keywords": broader_title, "location": clean_location, "f_WT": clean_typeofjob},
    # 3. Search in the area with standard job listings
    {"keywords": "Artificial Intelligence", "location": clean_location, "f_WT": clean_typeofjob},
    # 4. Global match in the country
    {"keywords": clean_job_title, "location": clean_location, "f_WT": ""}
]

job_details = None

print(f"Searching live LinkedIn job postings for '{clean_job_title}' in '{clean_location}' ({job_type_str})...")

for strategy in search_strategies:
    search_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={strategy['keywords']}&location={strategy['location']}&f_WT={strategy['f_WT']}"
    try:
        search_response = requests.get(search_url, headers=headers, timeout=10)
        search_soup = BeautifulSoup(search_response.text, 'html.parser')
        job_cards = search_soup.find_all('li')

        valid_cards = []
        if job_cards:
            for card in job_cards:
                job_id_elem = card.find('div', {'data-entity-urn': True})
                if job_id_elem:
                    valid_cards.append(job_id_elem)

        if valid_cards:
            print(f"Found jobs using query: '{strategy['keywords']}' in {strategy['location']}. Selecting a match...")
            random.shuffle(valid_cards)

            for selected_card in valid_cards:
                urn = selected_card['data-entity-urn']
                job_id = urn.split(':')[-1]
                direct_job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

                try:
                    detail_response = requests.get(direct_job_url, headers=headers, timeout=10)
                    detail_soup = BeautifulSoup(detail_response.text, 'html.parser')

                    title = detail_soup.find('h2', {'class': 'top-card-layout__title'})
                    company = detail_soup.find('a', {'class': 'topcard__org-name-link'})
                    desc = detail_soup.find('div', {'class': 'show-more-less-html__markup'})

                    parsed_desc = desc.get_text(separator='\n', strip=True) if desc else ""

                    # Guarantee check: Description must have actual content
                    if parsed_desc and len(parsed_desc) > 50:
                        job_details = {
                            "job_title": title.get_text(strip=True) if title else strategy['keywords'],
                            "location": clean_location,
                            "company": company.get_text(strip=True) if company else "Unknown Company",
                            "job_description": parsed_desc,
                            "job_type": job_type_str if strategy['f_WT'] else "Hybrid/Onsite",
                            "source": f"https://www.linkedin.com/jobs/view/{job_id}"
                        }
                        print(f"Successfully parsed job: '{job_details['job_title']}' at '{job_details['company']}'")
                        break
                except Exception:
                    continue

            if job_details:
                break
    except Exception:
        continue

# ABSOLUTE GUARANTEE FALLBACK: Generate customized data if LinkedIn fails entirely
if not job_details:
    print("LinkedIn limit reached or no listings found. Generating customized matching description to guarantee output...")
    fallback_companies = ["Apex Global Operations", "Vanguard Services", "Premier Holdings", "Interstate Group"]
    selected_fallback_company = random.choice(fallback_companies)
    job_details = {
        "job_title": f"{clean_job_title} ({job_type_str})",
        "location": clean_location,
        "company": selected_fallback_company,
        "job_description": f"We are seeking a motivated {clean_job_title} to join our team in {clean_location}. "
                           f"In this {job_type_str.lower()} role, you will work on cutting-edge systems, train models, assist with "
                           f"data analysis, and collaborate with engineering teams to deploy AI solutions.",
        "job_type": job_type_str,
        "source": "Generated Fallback Match"
    }
    print(f"Successfully generated match: '{job_details['job_title']}' at '{job_details['company']}'")

# Convert job details dict to binary bytes stream (JSON file representation)
binary_data = json.dumps(job_details, indent=4).encode('utf-8')
file_name = "job_description.json"

# Send the structured job details as raw binary to n8n
print(f"Sending job details to n8n workflow as a binary stream ({file_name})...")
try:
    response = requests.post(
        N8N_JOB_WEBHOOK_URL,
        data=binary_data,
        headers={
            "Content-Type": "application/octet-stream",
            "X-File-Name": file_name
        },
        timeout=30
    )
    if response.status_code == 200:
        print("Success! Real-time job description binary sent to n8n.")
        print("Response from n8n:", response.text)
    else:
        print(f"n8n returned status code: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"Failed to deliver to n8n webhook: {e}")

Searching live LinkedIn job postings for 'Computer Science' in 'London' (Remote)...
Found jobs using query: 'Computer Science' in London. Selecting a match...
Successfully parsed job: 'Backend Software Engineer C# Asset Management' at 'Client Server'
Sending job details to n8n workflow as a binary stream (job_description.json)...
Success! Real-time job description binary sent to n8n.
Response from n8n: {"message":"Workflow was started"}


CV Parsing

In [60]:
import os
import requests
from google.colab import files

# 1. Ask the user to upload their CV file
print("Please select and upload your CV (PDF, DOCX, etc.):")
uploaded = files.upload()

# 2. Configure your n8n webhook URL
N8N_WEBHOOK_URL = "https://bagged-attest-snooper.ngrok-free.dev/webhook/cv-upload"

binary_data = None
file_name = ""

# 3. Process the file (either newly uploaded or fallback to existing)
if uploaded:
    for name in uploaded.keys():
        file_name = name
        binary_data = uploaded[name]
        print(f"Using newly uploaded file: {file_name}")
else:
    # Fallback to an existing CV in the environment if the upload dialog was closed
    fallback_path = "/content/Abbas Jamil CV.docx (6).pdf"
    if os.path.exists(fallback_path):
        file_name = os.path.basename(fallback_path)
        with open(fallback_path, "rb") as f:
            binary_data = f.read()
        print(f"No file uploaded. Falling back to existing file in workspace: {file_name}")
    else:
        print("No file was uploaded and no fallback file was found.")

# 4. Send the file as raw binary to n8n
if binary_data:
    print(f"Sending {file_name} to n8n webhook as a binary stream...")
    try:
        # Send POST request with the raw binary data in the body
        response = requests.post(
            N8N_WEBHOOK_URL,
            data=binary_data,
            headers={
                "Content-Type": "application/octet-stream",
                "X-File-Name": file_name
            },
            timeout=60
        )

        if response.status_code == 200:
            print("Success! Webhook triggered successfully.")
            print("Response from n8n:", response.text)
        else:
            print(f"Failed to send to webhook. Status code: {response.status_code}")
            print("Response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Please select and upload your CV (PDF, DOCX, etc.):


Saving Laminy Sonko Sadio.pdf to Laminy Sonko Sadio (2).pdf
Using newly uploaded file: Laminy Sonko Sadio (2).pdf
Sending Laminy Sonko Sadio (2).pdf to n8n webhook as a binary stream...
Success! Webhook triggered successfully.
Response from n8n: {"message":"Workflow was started"}


Retrieving Job Description and CV Extracted data from n8n back into google colab

In [61]:
import os
import json
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the target filename we want to retrieve and delete
target_filename = 'job_description.json'
loaded_job_details = None
file_found_path = None

# Traverse Google Drive to locate the file
print("Searching for 'job_description.json' in your Google Drive...")
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if target_filename in files:
        file_found_path = os.path.join(root, target_filename)
        break

if file_found_path:
    print(f"Found file at: {file_found_path}")
    try:
        # Load the content from Google Drive
        with open(file_found_path, 'r', encoding='utf-8') as f:
            loaded_job_details = json.load(f)
        print("Successfully loaded data into 'loaded_job_details' variable.")

        # Delete the file from Google Drive now that it is stored in the variable
        os.remove(file_found_path)
        print(f"Successfully deleted '{file_found_path}' from Google Drive.")
    except Exception as e:
        print(f"An error occurred while reading or deleting the file: {e}")
else:
    print(f"Could not find '{target_filename}' on Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching for 'job_description.json' in your Google Drive...
Found file at: /content/drive/MyDrive/job_description.json
Successfully loaded data into 'loaded_job_details' variable.
Successfully deleted '/content/drive/MyDrive/job_description.json' from Google Drive.


In [62]:
import os

target_filename = 'job_description.json'
deleted_paths = []

print("Initiating thorough deep-search for duplicate files in Google Drive...")
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if target_filename in files:
        file_path = os.path.join(root, target_filename)
        try:
            os.remove(file_path)
            deleted_paths.append(file_path)
            print(f"Permanently deleted: {file_path}")
        except Exception as e:
            print(f"Could not delete {file_path}: {e}")

if not deleted_paths:
    print(f"No remaining files named '{target_filename}' were found on Google Drive.")
else:
    print(f"Cleanup complete. Deleted {len(deleted_paths)} instance(s) of '{target_filename}'.")

Initiating thorough deep-search for duplicate files in Google Drive...
No remaining files named 'job_description.json' were found on Google Drive.


In [63]:
import os
from google.colab import drive

# Mount Google Drive (if not already mounted)
drive.mount('/content/drive', force_remount=False)

target_cv_filename = 'CVdata.txt'
loaded_cv_data = None
cv_file_found_path = None

# Traverse Google Drive to locate CVdata.txt
print("Searching for 'CVdata.txt' in your Google Drive...")
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if target_cv_filename in files:
        cv_file_found_path = os.path.join(root, target_cv_filename)
        break

if cv_file_found_path:
    print(f"Found file at: {cv_file_found_path}")
    try:
        # Load the content from Google Drive
        with open(cv_file_found_path, 'r', encoding='utf-8') as f:
            loaded_cv_data = f.read()
        print("Successfully loaded data into 'loaded_cv_data' variable.")

        # Delete the file from Google Drive now that it is stored in the variable
        os.remove(cv_file_found_path)
        print(f"Successfully deleted '{cv_file_found_path}' from Google Drive.")
    except Exception as e:
        print(f"An error occurred while reading or deleting the file: {e}")
else:
    print(f"Could not find '{target_cv_filename}' on Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching for 'CVdata.txt' in your Google Drive...
Found file at: /content/drive/MyDrive/CVdata.txt
Successfully loaded data into 'loaded_cv_data' variable.
Successfully deleted '/content/drive/MyDrive/CVdata.txt' from Google Drive.


In [64]:
print(loaded_job_details)

[{'job_description': "Backend Software Engineer / Developer (C# .Net Claude) London / WFH to £100k\nAre you a backend focussed Software Engineer with strong C# skills combined with an outstanding academic record?\nYou could be progressing your career in a hands-on Backend Software Engineer role at a tech driven, global Asset Manager with multiple billions of dollars under management, working on complex and interesting systems with continual learning and development opportunities.\nAs a Backend Software Engineer you will join a small team to design and develop new features and enhancements to the client portal and KYC systems. You'll take ownership of the full software development lifecycle, designing, building and maintaining applications that support critical business processes, from initial development through to production.\nYou'll also have opportunities to explore emerging technologies and innovative approaches, including how AI can transform workflows and improve software develop

Skills Extraaction and Comparison

In [65]:
import re

def extract_skills_vocab(text):
    if not text:
        return set(), set()

    text_lower = text.lower()

    # Extended dictionary of common technical terms/tools
    tech_keywords = {
        'python', 'java', 'javascript', 'html', 'css', 'sql', 'git', 'github', 'botpress',
        'n8n', 'node', 'react', 'c#', '.net', 'c++', 'aws', 'docker', 'kubernetes', 'terraform',
        'apis', 'nlp', 'nlp techniques', 'prompt engineering', 'ai', 'artificial intelligence',
        'machine learning', 'deep learning', 'statistics', 'cloud', 'database', 'testing',
        'ci/cd', 'tdd', 'solid', 'devops', 'agile', 'scrum', 'networks', 'algorithms'
    }

    # Extended dictionary of professional soft skills and methodologies
    soft_keywords = {
        'collaborative', 'collaboration', 'teamwork', 'communication', 'problem solving',
        'leadership', 'time management', 'presentation', 'analytical', 'adaptability',
        'organization', 'mentorship', 'interpersonal', 'strategy', 'planning', 'motivation',
        'motivated', 'initiative', 'creative', 'critical thinking', 'decision making'
    }

    extracted_tech = set()
    extracted_soft = set()

    # Extract skills present in text
    for skill in tech_keywords:
        if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            extracted_tech.add(skill)

    for skill in soft_keywords:
        if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            extracted_soft.add(skill)

    return extracted_tech, extracted_soft

# Retrieve texts dynamically
job_desc_text = ""
if 'job_details' in globals() and job_details:
    job_desc_text = job_details.get('job_description', '')
elif 'loaded_job_details' in globals() and loaded_job_details:
    if isinstance(loaded_job_details, list) and len(loaded_job_details) > 0:
        job_desc_text = loaded_job_details[0].get('job_description', '')
    elif isinstance(loaded_job_details, dict):
        job_desc_text = loaded_job_details.get('job_description', '')

cv_text = loaded_cv_data if 'loaded_cv_data' in globals() and loaded_cv_data else ""

# Extract skills from CV
cv_tech, cv_soft = extract_skills_vocab(cv_text)
all_cv_skills = cv_tech.union(cv_soft)

# Extract skills from Job Description
job_tech, job_soft = extract_skills_vocab(job_desc_text)
all_job_skills = job_tech.union(job_soft)

# Perform matches on the combined skill set
matched_skills = all_job_skills.intersection(all_cv_skills)
missing_skills = all_job_skills.difference(all_cv_skills)

# Helper to format output display name
def format_skill(s):
    return s.upper() if s in ['n8n', 'apis', 'html', 'css', 'sql', 'nlp', 'aws', 'ci/cd', 'tdd', 'solid'] else s.title()

# Score calculation
match_score = (len(matched_skills) / len(all_job_skills) * 100) if all_job_skills else 0.0

# Output clean comprehensive report
print("         EXTRACTED CV SKILLS PROFILE              ")
print(f"💻 Your Technical Skills ({len(cv_tech)}):\n  {sorted([format_skill(s) for s in cv_tech])}")
print(f"🤝 Your Soft Skills ({len(cv_soft)}):\n  {sorted([format_skill(s) for s in cv_soft])}")
print("             CV vs JD COMPARISON REPORT           ")
print(f"Target Job Title: {job_details.get('job_title', 'Target Post') if 'job_details' in globals() else 'Job Post'}")
print(f"\nRequired Skills in Job Description: {sorted([format_skill(s) for s in all_job_skills])}")
print(f"\n✅ MATCHED SKILLS: {sorted([format_skill(s) for s in matched_skills]) if matched_skills else 'None'}")
print(f"❄ MISSING SKILLS: {sorted([format_skill(s) for s in missing_skills]) if missing_skills else 'None'}")
print(f"\n📊 OVERALL SKILLS MATCH SCORE: {match_score:.1f}%")

         EXTRACTED CV SKILLS PROFILE              
💻 Your Technical Skills (10):
  ['Algorithms', 'CSS', 'Database', 'Git', 'Github', 'HTML', 'Javascript', 'Networks', 'Python', 'SQL']
🤝 Your Soft Skills (2):
  ['Communication', 'Teamwork']
             CV vs JD COMPARISON REPORT           
Target Job Title: Backend Software Engineer C# Asset Management

Required Skills in Job Description: ['APIS', 'AWS', 'Ai', 'Collaborative', 'Communication', 'Kubernetes', 'React', 'SQL']

✅ MATCHED SKILLS: ['Communication', 'SQL']
❄ MISSING SKILLS: ['APIS', 'AWS', 'Ai', 'Collaborative', 'Kubernetes', 'React']

📊 OVERALL SKILLS MATCH SCORE: 25.0%
